In [2]:
import pandas as pd
from pandasql import sqldf
from pathlib import Path

In [3]:
pysqldf = lambda q: sqldf(q, globals())
caminho = Path.cwd()


In [4]:
buyers = pd.read_csv(f'{caminho}/dados/buyers.csv', sep=',')
order_items = pd.read_csv(f'{caminho}/dados/order_items.csv', sep=',')
orders = pd.read_csv(f'{caminho}/dados/orders.csv', sep=',')
payments = pd.read_csv(f'{caminho}/dados/payments.csv', sep=',')
products = pd.read_csv(f'{caminho}/dados/products.csv', sep=',')
sellers = pd.read_csv(f'{caminho}/dados/sellers.csv', sep=',')

In [5]:
display(buyers.head())
display(order_items.head())
display(orders.head())
display(payments.head())
display(products.head())
display(sellers.head())


,id,name,city,state,segment,created_at
0,1,Nascimento & Costa Distribuidora Mercearia,Recife,DF,supermarket,2023-04-03 03:46:36
1,2,Lima & Almeida Atacado Mercearia,Rio de Janeiro,PE,convenience,2023-08-05 01:36:12
2,3,Ferreira & Costa Grupo Mercearia,São Paulo,PR,grocery,2023-07-10 11:37:01
3,4,Lima & Almeida Comércio Supermercado,Campo Grande,MT,convenience,2023-10-13 01:08:52
4,5,Ferreira & Ferreira Comércio Mercearia,São Paulo,PB,convenience,2024-04-20 00:23:27


,id,order_id,product_id,qty,unit_price,discount
0,1,1,500,6,240.06,117.48
1,2,1,778,6,126.47,94.54
2,3,2,346,39,492.08,98.86
3,4,2,248,20,278.55,588.33
4,5,3,701,9,298.75,272.33


,id,seller_id,buyer_id,status,created_at,total_value
0,1,114,2806,delivered,2023-08-22 05:25:59,1987.16
1,2,90,2586,processing,2024-07-19 03:57:54,24074.93
2,3,96,1849,completed,2024-06-14 22:16:15,2416.42
3,4,10,116,processing,2024-07-11 04:46:30,3871.48
4,5,16,2195,delivered,2023-09-07 20:06:40,15367.65


,id,order_id,paid_at,amount,method,status
0,1,1,2023-08-23 19:25:59,1987.16,transfer,paid
1,2,2,2024-07-20 17:57:54,24074.93,boleto,paid
2,3,3,2024-06-16 10:16:15,2416.42,transfer,paid
3,4,4,2024-07-11 11:46:30,3871.48,boleto,paid
4,5,5,2023-09-08 00:06:40,15367.65,transfer,paid


,id,name,category,seller_id,active,unit_cost
0,1,Produto Laticínios Linha 1,Snacks,31,1,104.39
1,2,Produto Grãos Linha 2,Carnes,71,1,153.19
2,3,Produto Enlatados Linha 3,Grãos,9,0,90.57
3,4,Produto Snacks Linha 4,Bebidas,13,1,158.39
4,5,Produto Carnes Linha 5,Grãos,58,1,124.43


,id,name,state,plan,created_at
0,1,Santos & Silva Atacado Distribuidora,BA,free,2023-04-17 00:16:11
1,2,Souza & Souza Comércio Distribuidora,DF,premium,2022-09-09 22:06:21
2,3,Santos & Almeida Distribuidora Distribuidora,AM,basic,2022-12-16 08:31:25
3,4,Nascimento & Ferreira Alimentos Distribuidora,GO,basic,2023-05-30 23:12:37
4,5,Silva & Santos Suprimentos Distribuidora,BA,premium,2023-03-06 10:29:33


In [6]:
buyers.info()
order_items.info()
orders.info()
payments.info()
products.info()
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3001 entries, 0 to 3000
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          3001 non-null   int64
 1   name        3001 non-null   str  
 2   city        3001 non-null   str  
 3   state       3001 non-null   str  
 4   segment     3001 non-null   str  
 5   created_at  3001 non-null   str  
dtypes: int64(1), str(5)
memory usage: 140.8 KB
<class 'pandas.DataFrame'>
RangeIndex: 214768 entries, 0 to 214767
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          214768 non-null  int64  
 1   order_id    214768 non-null  int64  
 2   product_id  214768 non-null  int64  
 3   qty         214768 non-null  int64  
 4   unit_price  214768 non-null  float64
 5   discount    214768 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 9.8 MB
<class 'pandas.DataFrame'>
RangeIndex: 80002 entries, 0 to 8

In [7]:
## Foi encontrado dois IDS iguais em uma tabela de registros, afim de
## garantir a consistencia dos dados, sera necessario avaliar esses registros
# orders['id'].value_counts()
# display((orders[(orders['id'] == 79990) | (orders['id'] == 79991)]))
orders = orders[(orders['id'] != 79990) | (orders['id'] != 79991)]

# order_items['id'].value_counts()

## Foi encontrado dois IDS iguais em uma tabela de registros, afim de
## garantir a consistencia dos dados, sera necessario avaliar esses registros
# buyers['id'].value_counts()
# display((buyers[(buyers['id'] == 172)]))
buyers = buyers[(buyers['id'] != 172)]

# payments['id'].value_counts()

# products['id'].value_counts()

# sellers['id'].value_counts()


In [8]:
# Desafio 1
query = """
SELECT
    strftime('%Y-%m', A.created_at) AS mes,
    SUM(A.total_value) AS faturamento,
    COUNT(A.id) AS qtd_pedidos,
    ROUND(AVG(A.total_value),2) AS ticket_medio
FROM orders AS A
WHERE
    (A.status IN ('completed', 'delivered'))
    AND A.created_at >= (
        SELECT date(MAX(created_at), '-12 months') 
        FROM orders
    )
GROUP BY mes
ORDER BY mes DESC
"""

resultado_desafio1 = pysqldf(query)
resultado_desafio1


,mes,faturamento,qtd_pedidos,ticket_medio
0,2024-11,56710238.63,3643,15566.91
1,2024-10,62493439.13,3863,16177.44
2,2024-09,60274462.50,3779,15949.84
3,2024-08,60102539.82,3743,16057.32
4,2024-07,59769406.95,3862,15476.28
5,2024-06,58797774.37,3644,16135.50
6,2024-05,60730315.72,3848,15782.31
7,2024-04,57779312.59,3653,15816.95
8,2024-03,61525913.64,3877,15869.46
9,2024-02,57926376.26,3628,15966.48


In [9]:
# Desafio 2
query = """
WITH PeriodosInteresse AS (
-- Essa parte identifica os 2 últimos trimestres que REALMENTE existem no banco
    SELECT DISTINCT (CAST(strftime('%Y', created_at) AS INT) * 10 + (strftime('%m', created_at) + 2) / 3) AS p_id
    FROM orders
    ORDER BY p_id DESC
    LIMIT 2
),
VendasFiltradas AS (
-- Essa tabela e responsavel por me retornar um resumo de cada vendedor, 
-- me retornando o GMV (Gross Merchandise Volume, ou Volume Bruto de Mercadorias), 
-- ano, trimestre e o percentual desse desconto
    SELECT 
        A.seller_id,
        (CAST(strftime('%Y', A.created_at) AS INT) * 10 + (strftime('%m', A.created_at) + 2) / 3) AS p_id,
        SUM(D.gmv_pedido) AS gmv,
        COUNT(A.id) AS qtd_pedidos
    FROM orders AS A
    JOIN (
        SELECT order_id, SUM(qty * unit_price) AS gmv_pedido
        FROM order_items GROUP BY order_id
    ) AS D ON D.order_id = A.id
    WHERE p_id IN (SELECT p_id FROM PeriodosInteresse)
    GROUP BY A.seller_id, p_id
    HAVING COUNT(A.id) >= 50 
)
SELECT 
    S.name AS seller,
    S.state AS estado,
    -- Trimestre mais Recente
    ROUND(SUM(CASE WHEN F.p_id = (SELECT MAX(p_id) FROM PeriodosInteresse) THEN gmv ELSE 0 END), 2) AS faturamento_atual,
    -- Trimestre anterior
    ROUND(SUM(CASE WHEN F.p_id = (SELECT MIN(p_id) FROM PeriodosInteresse) THEN gmv ELSE 0 END), 2) AS faturamento_anterior,
    -- Crescimento %
    ROUND((
        (SUM(CASE WHEN F.p_id = (SELECT MAX(p_id) FROM PeriodosInteresse) THEN gmv ELSE 0 END) /
         NULLIF(SUM(CASE WHEN F.p_id = (SELECT MIN(p_id) FROM PeriodosInteresse) THEN gmv ELSE 0 END), 0)
        ) - 1
    ) * 100, 2) AS pct_crescimento
FROM VendasFiltradas F
JOIN sellers S ON S.id = F.seller_id
GROUP BY S.name, S.state
-- Garante que o seller apareca duas vezes
HAVING COUNT(F.p_id) = 2 
ORDER BY pct_crescimento DESC
LIMIT 10;
"""

resultado_desafio2 = pysqldf(query)
resultado_desafio2

,seller,estado,faturamento_atual,faturamento_anterior,pct_crescimento
0,Costa & Ferreira Atacado Distribuidora,PE,1062816.05,1060030.70,0.26
1,Costa & Silva Alimentos Distribuidora,BA,1378525.99,1484494.08,-7.14
2,Rodrigues & Almeida Atacado Distribuidora,MG,1195199.91,1289406.63,-7.31
3,Santos & Silva Suprimentos Distribuidora,MS,1195778.66,1293404.73,-7.55
4,Nascimento & Souza Alimentos Distribuidora,SP,1319309.05,1451443.24,-9.10
5,Santos & Silva Mercado Distribuidora,MT,1220953.19,1345217.42,-9.24
6,Nascimento & Souza Alimentos Distribuidora,GO,1130941.64,1247183.38,-9.32
7,Rodrigues & Oliveira Atacado Distribuidora,RS,1054700.05,1192972.78,-11.59
8,Lima & Almeida Grupo Distribuidora,CE,1260496.85,1426463.30,-11.63
9,Costa & Santos Alimentos Distribuidora,MG,1125999.62,1281994.69,-12.17


In [10]:
# Desafio 3
query = """
WITH DescontosItem AS(
-- Essa tabela e responsavel por me retornar um resumo
-- de cada pedido, com o valor bruto, o desconto e o
-- percentual desse desconto
    SELECT
        A.order_id
        ,SUM(A.qty * A.unit_price) AS valor_bruto
        ,SUM(A.discount) AS desconto
        ,ROUND(SUM(discount) * 1.0 / SUM(qty * unit_price),2) AS pct_desconto
    FROM order_items AS A
    GROUP BY A.order_id
    HAVING pct_desconto > 0.4
    ORDER BY pct_desconto DESC
)

SELECT
    A.order_id AS pedido
    ,A.valor_bruto
    ,A.desconto
    ,A.pct_desconto
    ,C.name AS seller
    ,DATE(B.created_at) AS data
FROM DescontosItem AS A
LEFT JOIN orders AS B ON A.order_id = B.id
LEFT JOIN sellers AS C ON B.seller_id =  C.id
WHERE 
    B.status <> 'cancelled'
ORDER BY A.pct_desconto DESC
"""

resultado_desafio3 = pysqldf(query)
resultado_desafio3

,pedido,valor_bruto,desconto,pct_desconto,seller,data
0,72401,930.30,558.02,0.60,Costa & Oliveira Atacado Distribuidora,2024-01-21
1,71165,13385.60,8017.88,0.60,Santos & Almeida Suprimentos Distribuidora,2024-11-09
2,53817,1404.78,837.12,0.60,Costa & Costa Atacado Distribuidora,2024-01-02
3,53223,7970.40,4773.99,0.60,Costa & Costa Atacado Distribuidora,2024-05-06
4,44426,4336.66,2586.99,0.60,Costa & Costa Atacado Distribuidora,2024-04-08
...,...,...,...,...,...,...
857,2869,18621.36,7565.58,0.41,Souza & Ferreira Alimentos Distribuidora,2023-11-25
858,2623,4038.51,1666.30,0.41,Santos & Almeida Suprimentos Distribuidora,2024-08-12
859,2308,4237.65,1746.25,0.41,Costa & Costa Atacado Distribuidora,2024-03-10
860,1024,18223.61,7410.48,0.41,Santos & Almeida Suprimentos Distribuidora,2024-06-07


In [11]:
# Desafio 4 - Independente da quantidade de produtos na cesta
query = """
WITH VolumeAlto AS (
-- Essa tabela e responsavel por me retornar ids dos 
-- produtos que tiveram quantidade total vendida maior que 1000
    SELECT
        O.product_id
        ,P.name
        ,P.category
        ,SUM(O.qty) as qtd
    FROM order_items AS O
    LEFT JOIN products AS P ON O.product_id = P.id
    GROUP BY product_id
    HAVING SUM(qty) > 1000
),
ProdutosPremium AS (
-- Essa tabela e responsavel por me retornar os ids dos produtos
-- com maior preco unico dentro de um pedido
    SELECT DISTINCT
        A.product_id
    FROM (
        SELECT 
            order_id
            ,product_id
            ,unit_price
            ,ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY unit_price DESC) AS rnk
        FROM order_items
    ) AS A
    WHERE A.rnk = 1
)

SELECT
    *
FROM VolumeAlto AS V
WHERE 
    V.product_id NOT IN (SELECT * FROM ProdutosPremium)
"""

resultado_desafio4 = pysqldf(query)
resultado_desafio4

,product_id,name,category,qtd


In [12]:
# Desafio 4 - Necessario mais de um produto a cesta
query = """
WITH VolumeAlto AS (
-- Essa tabela e responsavel por me retornar ids dos 
-- produtos que tiveram quantidade total vendida maior que 1000
    SELECT
        O.product_id
        ,P.name
        ,P.category
        ,SUM(O.qty) as qtd
    FROM order_items AS O
    LEFT JOIN products AS P ON O.product_id = P.id
    GROUP BY product_id
    HAVING SUM(qty) > 1000
),
ProdutosPremium AS (
-- Essa tabela e responsavel por me retornar os ids dos produtos
-- com maior preco unico dentro de um pedido dos pedidos que possuem
-- mais de um produto
    SELECT DISTINCT 
        product_id
    FROM (
        SELECT 
            product_id
            ,order_id
            ,RANK() OVER(PARTITION BY order_id ORDER BY unit_price DESC) AS rnk
            ,COUNT(*) OVER(PARTITION BY order_id) AS total_itens_no_pedido
        FROM order_items
    ) AS Ranking
    WHERE rnk = 1 AND total_itens_no_pedido > 1
)

SELECT 
    *
FROM VolumeAlto AS V
WHERE V.product_id NOT IN (SELECT product_id FROM ProdutosPremium);
"""

resultado_desafio4 = pysqldf(query)
resultado_desafio4

,product_id,name,category,qtd
